In [1]:
import matplotlib.pyplot as plt
import numpy as np
import qim3d
import ome_zarr
import ome_zarr.writer
import shutil
from functools import partial
import dask

import imagecodecs
import imagecodecs.zarr as codecs
import zarr.codecs.numcodecs

from comptest.constants import DATASETS_RECONSTRUCTIONS_DIR
from comptest.types import ReconstructionGeometry
import zarr
import zarr.storage

%load_ext autoreload
%autoreload 2

In [2]:
codecs.register_codecs(verbose=True)

In [3]:
DS_NAME = "Canislupus_CZM_712"

DS_DIR = DATASETS_RECONSTRUCTIONS_DIR / "CIL_basic_binned_2x2" / DS_NAME
ds_np = qim3d.io.load(
    DS_DIR,
    contains=DS_NAME + "_",
)
ds_geometry = ReconstructionGeometry.load_json(DS_DIR / "geometry.json")

In [4]:
# Fix: https://github.com/qim-center/qim3d/issues/215
dask.config.set(scheduler="threads")

In [5]:
target_dtype = np.uint16
target_dtype_info = np.iinfo(target_dtype)

# Affine transform to fit in [DT.MIN, DT.MAX]
shift = np.min(ds_np)
data_range = np.max(ds_np) - shift
scale = (target_dtype_info.max - target_dtype_info.min) / data_range

# Clip to ensure no funny business with floating point rounding errors
ds_np_dt = np.clip(
    np.rint((ds_np - shift) * scale + target_dtype_info.min),
    target_dtype_info.min,
    target_dtype_info.max,
).astype(target_dtype)


In [6]:
ds_geometry = ReconstructionGeometry.load_json(DS_DIR / "geometry.json")

In [7]:
_jpegxl_base_kwargs = {
    "effort": 7,
    "transfer": imagecodecs.JPEGXL.TRANSFER_FUNCTION.LINEAR,
    "photometric": imagecodecs.JPEGXL.COLOR_SPACE.GRAY,
    # "level": 5,  # Setting this forces it to be lossless somewhat.
}

output_id = "OME-Zarr"
serializers = {
    "uncompressed": "auto",
    "jpegls_ll": codecs.Jpegls(level=0),
    "jpegls_nll_25": codecs.Jpegls(level=25),
    "jpegls_nll_50": codecs.Jpegls(level=50),
    "jpegxl_ll": codecs.Jpegxl(**_jpegxl_base_kwargs, distance=0.0, lossless=True),
    "jpegxl_l_0_1": codecs.Jpegxl(**_jpegxl_base_kwargs, distance=0.1),
    "jpegxl_l_0_5": codecs.Jpegxl(**_jpegxl_base_kwargs, distance=0.5),
    "jpegxl_l_1_0": codecs.Jpegxl(**_jpegxl_base_kwargs, distance=1.0),
}

scale_factors = {
    "no_scales": None,
    "scales_2_4_8_16": [
        {
            "x": factor,
            "y": factor,
            "z": factor,
        }
        for factor in [2, 4, 8, 16]
    ],
}


In [ ]:
output_id = "OME-Zarr_quality"
serializers = {
    # "uncompressed": "auto",
    #
    # "jpegls_ll_000": codecs.Jpegls(level=0),
    # "jpegls_nll_001": codecs.Jpegls(level=1),
    # "jpegls_nll_002": codecs.Jpegls(level=2),
    # "jpegls_nll_004": codecs.Jpegls(level=4),
    # "jpegls_nll_008": codecs.Jpegls(level=8),
    # "jpegls_nll_016": codecs.Jpegls(level=16),
    # "jpegls_nll_032": codecs.Jpegls(level=32),
    # "jpegls_nll_064": codecs.Jpegls(level=64),
    # "jpegls_nll_128": codecs.Jpegls(level=128),
    # "jpegls_nll_255": codecs.Jpegls(level=255),
    #
    # "jpegxl_ll": codecs.Jpegxl(**_jpegxl_base_kwargs, distance=0.0, lossless=True),
    "jpegxl_l_00_010": codecs.Jpegxl(**_jpegxl_base_kwargs, distance=00.010),
    "jpegxl_l_00_025": codecs.Jpegxl(**_jpegxl_base_kwargs, distance=00.025),
    "jpegxl_l_00_050": codecs.Jpegxl(**_jpegxl_base_kwargs, distance=00.050),
    "jpegxl_l_00_100": codecs.Jpegxl(**_jpegxl_base_kwargs, distance=00.100),
    "jpegxl_l_00_250": codecs.Jpegxl(**_jpegxl_base_kwargs, distance=00.250),
    "jpegxl_l_00_500": codecs.Jpegxl(**_jpegxl_base_kwargs, distance=00.500),
    "jpegxl_l_01_000": codecs.Jpegxl(**_jpegxl_base_kwargs, distance=01.000),
    "jpegxl_l_02_500": codecs.Jpegxl(**_jpegxl_base_kwargs, distance=02.500),
    "jpegxl_l_05_000": codecs.Jpegxl(**_jpegxl_base_kwargs, distance=05.000),
    "jpegxl_l_10_000": codecs.Jpegxl(**_jpegxl_base_kwargs, distance=10.000),
}

scale_factors = {
    "no_scales": None,
}

In [9]:
for serializer_name, serializer in serializers.items():
    for scale_factor_name, scale_factor in scale_factors.items():
        output_path = (
            DATASETS_RECONSTRUCTIONS_DIR
            / "CIL_basic_binned_2x2"
            / "NaturArv3D_format_viability_study"
            / output_id
            / f"{DS_NAME}__{serializer_name}__{scale_factor_name}.ome.zarr"
        )

        shutil.rmtree(output_path, ignore_errors=True)

        print(
            f"Writing {DS_NAME} to {output_path} with serializer {serializer_name} and scale factor {scale_factor_name}..."
        )
        ome_zarr.writer.write_image(
            ds_np_dt,
            str(output_path.resolve()),
            scale={
                "x": ds_geometry.voxel_size_x_m,
                "y": ds_geometry.voxel_size_y_m,
                "z": ds_geometry.voxel_size_z_m,
            },
            scale_factors=scale_factor,
            axes="zyx",
            axes_units={
                "x": "meter",
                "y": "meter",
                "z": "meter",
            },
            storage_options={
                "serializer": serializer,
                "chunks": (1, 1000, 1000),
                "shards": (50, 1000, 1000),
            },
        )


Writing Canislupus_CZM_712 to /dtu/3d-imaging-center/projects/2026_DANFIX_201/analysis/compression/datasets_reconstructions/CIL_basic_binned_2x2/NaturArv3D_format_viability_study/OME-Zarr_quality/Canislupus_CZM_712__jpegls_nll_001__no_scales.ome.zarr with serializer jpegls_nll_001 and scale factor no_scales...
Writing Canislupus_CZM_712 to /dtu/3d-imaging-center/projects/2026_DANFIX_201/analysis/compression/datasets_reconstructions/CIL_basic_binned_2x2/NaturArv3D_format_viability_study/OME-Zarr_quality/Canislupus_CZM_712__jpegls_nll_002__no_scales.ome.zarr with serializer jpegls_nll_002 and scale factor no_scales...
Writing Canislupus_CZM_712 to /dtu/3d-imaging-center/projects/2026_DANFIX_201/analysis/compression/datasets_reconstructions/CIL_basic_binned_2x2/NaturArv3D_format_viability_study/OME-Zarr_quality/Canislupus_CZM_712__jpegxl_l_00_010__no_scales.ome.zarr with serializer jpegxl_l_00_010 and scale factor no_scales...
Writing Canislupus_CZM_712 to /dtu/3d-imaging-center/projects/

In [10]:
from pprint import pprint

pprint(zarr.config.to_dict())

{'array': {'order': 'C',
           'read_missing_chunks': True,
           'rectilinear_chunks': False,
           'sharding_coalesce_max_bytes': 16777216,
           'sharding_coalesce_max_gap_bytes': 1048576,
           'target_shard_size_bytes': None,
           'write_empty_chunks': False},
 'async': {'concurrency': 10, 'timeout': None},
 'buffer': 'zarr.buffer.cpu.Buffer',
 'codec_pipeline': {'batch_size': 1,
                    'max_workers': None,
                    'path': 'zarr.core.codec_pipeline.BatchedCodecPipeline'},
 'codecs': {'blosc': 'zarr.codecs.blosc.BloscCodec',
            'bytes': 'zarr.codecs.bytes.BytesCodec',
            'crc32c': 'zarr.codecs.crc32c_.Crc32cCodec',
            'endian': 'zarr.codecs.bytes.BytesCodec',
            'gzip': 'zarr.codecs.gzip.GzipCodec',
            'numcodecs.adler32': 'zarr.codecs.numcodecs.Adler32',
            'numcodecs.astype': 'zarr.codecs.numcodecs.AsType',
            'numcodecs.bitround': 'zarr.codecs.numcodecs.BitRound

In [11]:
import dask
import dask.array as da
from dask.base import get_scheduler

probe_array = da.ones((4,), chunks=1)
effective_scheduler = get_scheduler(collections=[probe_array])

print("Dask version:", dask.__version__)
print("Configured scheduler:", dask.config.get("scheduler", None))
print("Configured worker count:", dask.config.get("num_workers", None))
print(
    "Multiprocessing context:",
    dask.config.get("multiprocessing.context", None),
)
print(
    "Effective scheduler:",
    f"{getattr(effective_scheduler, '__module__', '?')}."
    f"{getattr(effective_scheduler, '__qualname__', effective_scheduler)}",
)

try:
    from distributed import get_client

    client = get_client()
except (ImportError, ValueError):
    client = None

print("Active distributed Client:", client)

Dask version: 2026.7.1
Configured scheduler: threads
Configured worker count: None
Multiprocessing context: None
Effective scheduler: dask.threaded.get
Active distributed Client: None


In [12]:
import os
import threading
import time

import dask
from dask import delayed


@delayed(pure=False)
def execution_context(task_number):
    time.sleep(0.2)
    return {
        "task": task_number,
        "pid": os.getpid(),
        "thread_id": threading.get_ident(),
        "thread_name": threading.current_thread().name,
    }


contexts = dask.compute(*(execution_context(i) for i in range(8)))

print("Notebook PID:", os.getpid())
for context in contexts:
    print(context)

print("Distinct PIDs:", sorted({item["pid"] for item in contexts}))
print(
    "Distinct thread IDs:",
    sorted({item["thread_id"] for item in contexts}),
)

Notebook PID: 1005012
{'task': 0, 'pid': 1005012, 'thread_id': 140230114653760, 'thread_name': 'ThreadPoolExecutor-0_16'}
{'task': 1, 'pid': 1005012, 'thread_id': 140238547211840, 'thread_name': 'ThreadPoolExecutor-0_1'}
{'task': 2, 'pid': 1005012, 'thread_id': 140237208909376, 'thread_name': 'ThreadPoolExecutor-0_12'}
{'task': 3, 'pid': 1005012, 'thread_id': 140229260203584, 'thread_name': 'ThreadPoolExecutor-0_19'}
{'task': 4, 'pid': 1005012, 'thread_id': 140236706596416, 'thread_name': 'ThreadPoolExecutor-0_15'}
{'task': 5, 'pid': 1005012, 'thread_id': 140229224949312, 'thread_name': 'ThreadPoolExecutor-0_21'}
{'task': 6, 'pid': 1005012, 'thread_id': 140237986588224, 'thread_name': 'ThreadPoolExecutor-0_4'}
{'task': 7, 'pid': 1005012, 'thread_id': 140240629667392, 'thread_name': 'ThreadPoolExecutor-0_0'}
Distinct PIDs: [1005012]
Distinct thread IDs: [140229224949312, 140229260203584, 140230114653760, 140236706596416, 140237208909376, 140237986588224, 140238547211840, 140240629667392

In [13]:
import os

import dask
from dask import delayed

CODEC_KEY = "imagecodecs_jpegls"


def inspect_codec_registry():
    try:
        from zarr.registry import get_codec_class

        codec_class = get_codec_class(CODEC_KEY)
    except Exception as error:
        return {
            "pid": os.getpid(),
            "registered": False,
            "codec": None,
            "error": f"{type(error).__name__}: {error}",
        }

    return {
        "pid": os.getpid(),
        "registered": True,
        "codec": f"{codec_class.__module__}.{codec_class.__qualname__}",
        "error": None,
    }


print("Notebook registry:")
print(inspect_codec_registry())

worker_checks = dask.compute(
    *(delayed(inspect_codec_registry, pure=False)() for _ in range(8))
)

print("\nTask execution registries:")
for check in worker_checks:
    print(check)

Notebook registry:
{'pid': 1005012, 'registered': True, 'codec': 'imagecodecs.zarr.Jpegls', 'error': None}

Task execution registries:
{'pid': 1005012, 'registered': True, 'codec': 'imagecodecs.zarr.Jpegls', 'error': None}
{'pid': 1005012, 'registered': True, 'codec': 'imagecodecs.zarr.Jpegls', 'error': None}
{'pid': 1005012, 'registered': True, 'codec': 'imagecodecs.zarr.Jpegls', 'error': None}
{'pid': 1005012, 'registered': True, 'codec': 'imagecodecs.zarr.Jpegls', 'error': None}
{'pid': 1005012, 'registered': True, 'codec': 'imagecodecs.zarr.Jpegls', 'error': None}
{'pid': 1005012, 'registered': True, 'codec': 'imagecodecs.zarr.Jpegls', 'error': None}
{'pid': 1005012, 'registered': True, 'codec': 'imagecodecs.zarr.Jpegls', 'error': None}
{'pid': 1005012, 'registered': True, 'codec': 'imagecodecs.zarr.Jpegls', 'error': None}


In [14]:
import dask
from imagecodecs.zarr import register_codecs

register_codecs(verbose=False)

with dask.config.set(scheduler="threads"):
    worker_checks = dask.compute(
        *(delayed(inspect_codec_registry, pure=False)() for _ in range(8))
    )

for check in worker_checks:
    print(check)

{'pid': 1005012, 'registered': True, 'codec': 'imagecodecs.zarr.Jpegls', 'error': None}
{'pid': 1005012, 'registered': True, 'codec': 'imagecodecs.zarr.Jpegls', 'error': None}
{'pid': 1005012, 'registered': True, 'codec': 'imagecodecs.zarr.Jpegls', 'error': None}
{'pid': 1005012, 'registered': True, 'codec': 'imagecodecs.zarr.Jpegls', 'error': None}
{'pid': 1005012, 'registered': True, 'codec': 'imagecodecs.zarr.Jpegls', 'error': None}
{'pid': 1005012, 'registered': True, 'codec': 'imagecodecs.zarr.Jpegls', 'error': None}
{'pid': 1005012, 'registered': True, 'codec': 'imagecodecs.zarr.Jpegls', 'error': None}
{'pid': 1005012, 'registered': True, 'codec': 'imagecodecs.zarr.Jpegls', 'error': None}


In [15]:
import zarr.registry

zarr.registry._codec_registries

defaultdict(zarr.registry.Registry,
            {'blosc': {'zarr.codecs.blosc.BloscCodec': zarr.codecs.blosc.BloscCodec},
             'cast_value': {'zarr.codecs.cast_value.CastValue': zarr.codecs.cast_value.CastValue},
             'bytes': {'zarr.codecs.bytes.BytesCodec': zarr.codecs.bytes.BytesCodec},
             'endian': {'zarr.codecs.bytes.BytesCodec': zarr.codecs.bytes.BytesCodec},
             'crc32c': {'zarr.codecs.crc32c_.Crc32cCodec': zarr.codecs.crc32c_.Crc32cCodec},
             'gzip': {'zarr.codecs.gzip.GzipCodec': zarr.codecs.gzip.GzipCodec},
             'scale_offset': {'zarr.codecs.scale_offset.ScaleOffset': zarr.codecs.scale_offset.ScaleOffset},
             'sharding_indexed': {'zarr.codecs.sharding.ShardingCodec': zarr.codecs.sharding.ShardingCodec},
             'zstd': {'zarr.codecs.zstd.ZstdCodec': zarr.codecs.zstd.ZstdCodec},
             'vlen-utf8': {'zarr.codecs.vlen_utf8.VLenUTF8Codec': zarr.codecs.vlen_utf8.VLenUTF8Codec},
             'vlen-bytes': {'

In [16]:
ds_np_dt.shape

(1000, 1000, 1000)

In [17]:
store = zarr.storage.MemoryStore()
z = zarr.create_array(
    store=store,
    shape=ds_np_dt.shape,
    chunks=(3, 1000, 1000),
    shards="auto",
    dtype=ds_np_dt.dtype,
    serializer=codecs.Jpegxl(
        effort=5,
        distance=0.0,
        lossless=True,
        level=5,
    ),
    # compressor=codecs.Jpegxl(
    #     effort=1,
    #     distance=0.0,
    #     lossless=True,
    #     level=5,
    #     planar=True,
    #     transfer=imagecodecs.JPEGXL.TRANSFER_FUNCTION.LINEAR,
    #     photometric=imagecodecs.JPEGXL.COLOR_SPACE.GRAY,
    #     usecontainer=False,
    # ),
    config={
        "array.target_shard_size_bytes": 1024 * 1024  # 1 MB
    },
)

z[:] = ds_np_dt

/work3/s250250/pixi_cache/envs/comptest-2327749380986101308/envs/default/lib/python3.13/site-packages/zarr/core/array.py:4480: ZarrUserWarning: Automatic shard shape inference is experimental and may change without notice.
  outer_chunks, inner = resolve_outer_and_inner_chunks(


In [18]:
z.shards

(6, 1000, 1000)